# 用结构化数据生成基础图

## 节点构造

In [ ]:
# 构建全部在市 A 股的 N x 10 特征矩阵（遵循项目编码规范）
# 特征顺序：ln_cap, beta, momentum_20d, volatility_60d, turnover_20d, pe_ttm, pb, roe, debt_ratio, eps_growth
import os
import time
from datetime import datetime

import numpy as np
import pandas as pd

try:
    import tushare as ts
except Exception:
    raise ImportError('请先安装 tushare: pip install tushare')

try:
    import getpass
except Exception:
    getpass = None

FEATURE_COLUMNS = [
    'ln_cap', 'beta', 'momentum_20d', 'volatility_60d', 'turnover_20d',
    'pe_ttm', 'pb', 'roe', 'debt_ratio', 'eps_growth'
]


def get_token_manual_or_env():
    """获取 Tushare token，优先环境变量，否则交互输入（隐藏）。"""
    token = os.environ.get('TUSHARE_TOKEN')  # token 环境变量
    if not token:
        if getpass is not None:
            token = getpass.getpass('请输入你的 TUSHARE_TOKEN（输入时隐藏）：').strip()
        else:
            token = input('请输入你的 TUSHARE_TOKEN：').strip()
    if not token:
        print('错误：未提供 TUSHARE_TOKEN。请在环境变量中设置或在提示中输入。')
        raise RuntimeError('未提供 TUSHARE_TOKEN')
    print('✅ 已获取 TUSHARE_TOKEN（来源：环境变量或手动输入）')
    return token


def get_latest_trade_date_for_pro(pro):
    """返回最新交易日字符串，调用 Tushare 交易日历。"""
    cal = pro.trade_cal(exchange='', start_date='20000101', end_date=datetime.now().strftime('%Y%m%d'), is_open='1')
    cal = cal.sort_values('cal_date')
    latest_date = cal['cal_date'].iloc[-1]
    print(f'✅ 最近交易日为 {latest_date}')
    return latest_date


def fetch_stock_list_all(pro):
    """获取全部在市 A 股列表并返回 DataFrame。"""
    stocks = pro.stock_basic(exchange='', list_status='L', fields='ts_code,name,industry')
    stocks = stocks[stocks['ts_code'].str.endswith(('.SH', '.SZ'))].copy()
    stocks = stocks.sort_values('ts_code').reset_index(drop=True)
    if stocks.empty:
        print('错误：未能获取到在市股票列表，请检查网络或 token。')
        raise RuntimeError('股票列表为空')
    print(f'✅ 已获取在市股票列表，共 {len(stocks)} 支')
    return stocks


def fetch_snapshot_daily_basic(pro, trade_date):
    """获取指定交易日的 daily_basic 快照（市值、PE、PB、换手等）。"""
    try:
        db = pro.daily_basic(trade_date=trade_date, fields='ts_code,total_mv,turnover_rate,pe_ttm,pb')
        if db is None or db.empty:
            print('警告：daily_basic 返回空，部分快照字段将缺失')
            return pd.DataFrame()
        print('✅ daily_basic 快照已获取')
        return db.set_index('ts_code')
    except Exception as e:
        print('错误：获取 daily_basic 失败，请检查接口权限或网络。')
        raise


def fetch_market_returns(pro, trade_date, window=180):
    """获取市场（上证 000001.SH）近期收益，用于计算 beta。"""
    try:
        idx = pro.index_daily(ts_code='000001.SH', end_date=trade_date, limit=window)
        idx = idx.sort_values('trade_date')
        idx['mkt_ret'] = idx['close'].astype(float).pct_change()
        mkt_ret = idx[['trade_date', 'mkt_ret']].dropna()
        print('✅ 市场收益序列已获取')
        return mkt_ret
    except Exception:
        print('警告：获取市场收益失败，beta 将被填为 NaN')
        return pd.DataFrame()


def compute_feature_for_code(pro, stock_code, market_returns, snapshot_db, min_history=61):
    """为单只股票计算所需特征，返回字典（单行）。"""
    # 比喻注释：这里相当于服务员把菜单递给后厨，把股票历史和市场信息合并计算指标
    try:
        df = pro.daily(ts_code=stock_code, end_date=market_returns['trade_date'].iloc[-1] if not market_returns.empty else None, limit=180)
    except Exception:
        print(f'警告：拉取 {stock_code} 日线失败，跳过该只股票')
        return None

    if df is None or df.empty:
        print(f'警告：{stock_code} 无日线数据或数据为空，跳过')
        return None

    df = df.sort_values('trade_date').copy()
    df['close'] = df['close'].astype(float)
    df['ret'] = df['close'].pct_change()

    if len(df) < min_history:
        print(f'警告：{stock_code} 可用历史不足 {min_history} 日，跳过')
        return None

    # 计算 momentum 和 volatility（更稳健的实现）
    if len(df) >= 21 and pd.notna(df['close'].iloc[-21]) and df['close'].iloc[-21] != 0:
        momentum_20d = (df['close'].iloc[-1] - df['close'].iloc[-21]) / df['close'].iloc[-21]
    else:
        momentum_20d = np.nan

    last_returns = df['ret'].dropna().iloc[-60:]
    volatility_60d = float(last_returns.std(ddof=0)) if len(last_returns) >= 2 else np.nan

    beta_value = np.nan
    if not market_returns.empty:
        merged = df[['trade_date', 'ret']].merge(market_returns, on='trade_date', how='inner').dropna()
        merged = merged.tail(60)
        if len(merged) >= 20:
            var_m = float(np.var(merged['mkt_ret'].values))
            if var_m > 0:
                beta_value = float(np.cov(merged['ret'].values, merged['mkt_ret'].values, ddof=0)[0, 1] / var_m)

    # 从快照读取市值/换手/PE/PB（稳健处理）
    ln_cap = np.nan
    turnover_20d = np.nan
    pe_ttm = np.nan
    pb = np.nan
    if not snapshot_db.empty and stock_code in snapshot_db.index:
        try:
            raw_total_mv = snapshot_db.at[stock_code, 'total_mv']
            if pd.notna(raw_total_mv):
                try:
                    total_mv_float = float(raw_total_mv)
                    ln_cap = float(np.log(total_mv_float + 1.0))
                except Exception:
                    print(f'警告：{stock_code} total_mv 类型无法转换为数值，值={raw_total_mv}')
                    ln_cap = np.nan
            raw_turnover = snapshot_db.at[stock_code, 'turnover_rate']
            turnover_20d = float(raw_turnover) if pd.notna(raw_turnover) else np.nan
            raw_pe = snapshot_db.at[stock_code, 'pe_ttm']
            pe_ttm = float(raw_pe) if pd.notna(raw_pe) else np.nan
            raw_pb = snapshot_db.at[stock_code, 'pb']
            pb = float(raw_pb) if pd.notna(raw_pb) else np.nan
            print(f'✅ {stock_code} 快照字段读取完成')
        except KeyError:
            print(f'警告：快照中缺少 {stock_code} 的部分字段，已置为 NaN')
        except Exception as e:
            print(f'错误：处理 {stock_code} 快照时出错：{e}')

    return {
        'ts_code': stock_code,
        'ln_cap': ln_cap,
        'beta': beta_value,
        'momentum_20d': momentum_20d,
        'volatility_60d': volatility_60d,
        'turnover_20d': turnover_20d,
        'pe_ttm': pe_ttm,
        'pb': pb,
        'roe': np.nan,
        'debt_ratio': np.nan,
        'eps_growth': np.nan,
    }


def fetch_financials_for_codes(pro, codes, feats_df, sleep_interval=0.05):
    """逐股抓取财务指标并填充到 feats_df（roe, debt_ratio, eps_growth）。"""
    print('⏳ 开始逐股抓取财务指标（roe, debt_ratio, eps_growth）')
    for code in codes:
        try:
            fi = pro.fina_indicator(ts_code=code, limit=1, fields='ts_code,roe,debt_to_assets,basic_eps_yoy')
            if fi is not None and not fi.empty:
                row = fi.iloc[0]
                try:
                    feats_df.at[code, 'roe'] = float(row.get('roe')) if pd.notna(row.get('roe')) else np.nan
                except Exception:
                    feats_df.at[code, 'roe'] = np.nan
                    print(f'警告：{code} roe 无法转换为数值')
                try:
                    feats_df.at[code, 'debt_ratio'] = float(row.get('debt_to_assets')) if pd.notna(row.get('debt_to_assets')) else np.nan
                except Exception:
                    feats_df.at[code, 'debt_ratio'] = np.nan
                    print(f'警告：{code} debt_to_assets 无法转换为数值')
                try:
                    feats_df.at[code, 'eps_growth'] = float(row.get('basic_eps_yoy')) if pd.notna(row.get('basic_eps_yoy')) else np.nan
                except Exception:
                    feats_df.at[code, 'eps_growth'] = np.nan
                    print(f'警告：{code} basic_eps_yoy 无法转换为数值')
                print(f'✅ {code} 财务指标已填充')
            else:
                print(f'⚠️ {code} 无财务指标，已置为 NaN')
        except Exception as e:
            print(f'警告：抓取 {code} 财务指标失败：{e}')
        time.sleep(sleep_interval)
    print('✅ 全部财务指标抓取完成')


def build_all_feature_matrix(out_dir='data', sleep_interval=0.05, fillna_value=0.0, fetch_fina=False):
    """主流程：按步骤获取数据并计算全部股票的 10 维特征矩阵。"""
    token = get_token_manual_or_env()
    pro = ts.pro_api(token)

    stocks = fetch_stock_list_all(pro)
    codes = stocks['ts_code'].tolist()  # 股票代码列表
    latest_date = get_latest_trade_date_for_pro(pro)

    snapshot_db = fetch_snapshot_daily_basic(pro, latest_date)
    market_returns = fetch_market_returns(pro, latest_date)

    rows = []
    for code in codes:
        feat = compute_feature_for_code(pro, code, market_returns, snapshot_db)
        if feat is not None:
            rows.append(feat)
        time.sleep(sleep_interval)

    feats_df = pd.DataFrame(rows).drop_duplicates('ts_code').set_index('ts_code') if rows else pd.DataFrame(columns=FEATURE_COLUMNS)
    feats_df = feats_df.reindex(codes)

    if fetch_fina:
        # 使用独立函数逐股补齐财务指标
        fetch_financials_for_codes(pro, codes, feats_df, sleep_interval=sleep_interval)

    feats_df = feats_df[FEATURE_COLUMNS]
    feats_df = feats_df.replace([np.inf, -np.inf], np.nan).fillna(fillna_value)

    os.makedirs(out_dir, exist_ok=True)
    feats_df.to_csv(os.path.join(out_dir, 'node_feats_allx10.csv'))
    pd.DataFrame({'ts_code': codes}).to_csv(os.path.join(out_dir, 'stock_pool_all.csv'), index=False)
    feats_matrix = feats_df.to_numpy(dtype=np.float32)
    np.save(os.path.join(out_dir, 'node_feats_allx10.npy'), feats_matrix)

    print(f'✅ 完成：特征矩阵形状 = {feats_df.shape}')
    print('保存路径：', os.path.join(out_dir, 'node_feats_allx10.csv'))
    return feats_df, feats_matrix

# 兼容旧名字
build_5000_feature_matrix = build_all_feature_matrix


In [2]:
# 执行抓取并生成全部在市 A 股的 Nx10 特征矩阵（N 约 5000）
# 建议第一次运行不补财务指标以节省时间
try:
    feats_df, feats_matrix = build_all_feature_matrix(
        out_dir='data',
        sleep_interval=0.05,
        fillna_value=0.0,
        fetch_fina=False,
    )
    print('matrix shape:', feats_matrix.shape)
    print('✅ 已生成特征矩阵，前 5 行预览：')
    display(feats_df.head())
except Exception as e:
    print('错误：生成特征矩阵失败。请查看上方错误信息并重试。')
    raise


✅ 已获取 TUSHARE_TOKEN（来源：环境变量或手动输入）
✅ 已获取在市股票列表，共 5207 支
✅ 最近交易日为 20260522
✅ daily_basic 快照已获取
✅ 市场收益序列已获取
警告：001237.SZ 可用历史不足 61 日，跳过
警告：001257.SZ 可用历史不足 61 日，跳过
警告：001312.SZ 可用历史不足 61 日，跳过
警告：001365.SZ 可用历史不足 61 日，跳过
警告：001393.SZ 可用历史不足 61 日，跳过
警告：301513.SZ 可用历史不足 61 日，跳过
警告：301531.SZ 可用历史不足 61 日，跳过
警告：301599.SZ 可用历史不足 61 日，跳过
警告：301666.SZ 可用历史不足 61 日，跳过
警告：301680.SZ 可用历史不足 61 日，跳过
警告：301682.SZ 可用历史不足 61 日，跳过
警告：301683.SZ 可用历史不足 61 日，跳过
警告：301696.SZ 可用历史不足 61 日，跳过
警告：603293.SH 可用历史不足 61 日，跳过
警告：603407.SH 可用历史不足 61 日，跳过
警告：603435.SH 可用历史不足 61 日，跳过
警告：603459.SH 可用历史不足 61 日，跳过
警告：688781.SH 可用历史不足 61 日，跳过
警告：688808.SH 可用历史不足 61 日，跳过
警告：688811.SH 可用历史不足 61 日，跳过
警告：688813.SH 可用历史不足 61 日，跳过
警告：688820.SH 可用历史不足 61 日，跳过
✅ 完成：特征矩阵形状 = (5207, 10)
保存路径： data/node_feats_allx10.csv
matrix shape: (5207, 10)
✅ 已生成特征矩阵，前 5 行预览：


,ln_cap,beta,momentum_20d,volatility_60d,turnover_20d,pe_ttm,pb,roe,debt_ratio,eps_growth
ts_code,,,,,,,,,,
000001.SZ,16.846876,0.520154,-0.036101,0.010511,0.4917,4.8132,0.4466,0.0,0.0,0.0
000002.SZ,15.233310,1.503183,-0.112821,0.019310,0.8918,0.0000,0.3735,0.0,0.0,0.0
000004.SZ,0.000000,0.061936,-0.452381,0.040553,0.0000,0.0000,0.0000,0.0,0.0,0.0
000006.SZ,14.199953,1.969739,0.125129,0.035460,3.5696,0.0000,2.7357,0.0,0.0,0.0
000007.SZ,12.975429,1.633988,-0.092498,0.037995,8.9416,0.0000,24.1034,0.0,0.0,0.0


In [ ]:
# 从已保存的特征文件补齐财务指标（roe, debt_ratio, eps_growth）并覆盖保存
# 说明：依赖本 notebook 中已定义的 `get_token_manual_or_env` 和 `fetch_financials_for_codes` 函数
import os
import time
import shutil
import numpy as np
import pandas as pd
from datetime import datetime


def _load_feats(out_dir='data'):
    path = os.path.join(out_dir, 'node_feats_allx10.csv')
    if not os.path.exists(path):
        print('错误：未找到已生成的特征文件（node_feats_allx10.csv），请先运行生成特征的单元。')
        return None, path
    df = pd.read_csv(path, index_col=0)
    print(f'✅ 读取特征文件，行数={len(df)}')
    return df, path


def _save_feats(out_dir, feats_df):
    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, 'node_feats_allx10.csv')
    npy_path = os.path.join(out_dir, 'node_feats_allx10.npy')
    feats_df.to_csv(csv_path)
    matrix = feats_df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)
    np.save(npy_path, matrix)
    print(f'✅ 已保存 CSV -> {csv_path}，并更新 NPY -> {npy_path}')


def _backup_file(path):
    try:
        if os.path.exists(path):
            bak = f"{path}.bak.{datetime.now().strftime('%Y%m%d%H%M%S')}"
            shutil.copy2(path, bak)
            print(f'✅ 已备份原文件到 {bak}')
    except Exception as e:
        print(f'警告：备份原文件失败：{e}')


# 主流程
try:
    print('开始：使用 Tushare 补齐财务指标（roe, debt_ratio, eps_growth）到已有文件')
    feats, csv_path = _load_feats('data')
    if feats is None:
        raise RuntimeError('缺少输入文件')

    # 确保包含目标列
    for c in FEATURE_COLUMNS:
        if c not in feats.columns:
            feats[c] = np.nan

    # 只更新缺失的股票，避免全部重抓
    mask_missing = feats[ ['roe', 'debt_ratio', 'eps_growth'] ].isna().any(axis=1)
    codes_to_update = feats.index[mask_missing].tolist()
    if not codes_to_update:
        print('信息：没有检测到缺失的财务指标，跳过抓取。')
    else:
        print(f'⏳ 发现 {len(codes_to_update)} 支股票需要补齐财务指标（建议分批执行以节省并发限制）')

        token = get_token_manual_or_env()
        import tushare as ts
        pro = ts.pro_api(token)

        # 备份文件以防意外
        _backup_file(csv_path)

        # 分批抓取以降低单次压力（每批 200 支）
        batch_size = 200
        for i in range(0, len(codes_to_update), batch_size):
            batch = codes_to_update[i:i+batch_size]
            print(f'→ 抓取批次 {i//batch_size + 1}: {len(batch)} 支（索引 {i} - {i+len(batch)-1}）')
            fetch_financials_for_codes(pro, batch, feats, sleep_interval=0.05)
            # 保存中间结果以免丢失
            _save_feats('data', feats)
            time.sleep(0.5)

        print('✅ 所有需要的财务指标已尝试抓取并保存')

    # 最终保存并生成 NPY
    _save_feats('data', feats)
    print('完成：已在原文件中补齐财务指标')
except KeyboardInterrupt:
    print('操作被用户中断，已保存到最近的中间结果（如有）。')
except Exception as e:
    print('错误：补齐财务指标过程中发生异常，请查看日志。')
    raise


开始：使用 Tushare 补齐财务指标（roe, debt_ratio, eps_growth）到已有文件
✅ 读取特征文件，行数=5207
✅ 已获取 TUSHARE_TOKEN（来源：环境变量或手动输入）
⏳ 将抓取 5207 支股票的财务指标（可能较慢，建议使用较小 sleep_interval）
错误：补齐财务指标过程中发生异常，请查看日志。


NameError: name 'fetch_financials_for_codes' is not defined